# On-Chain Pool Data — FinTech 590
Queries live Uniswap V3 pool state directly from each chain using the saved ABIs.

**Run order:** `defi_pipeline.ipynb` → `historical_data.ipynb` → `database.ipynb` → this notebook.

**What this fetches per pool:**
| Field | Source | Description |
|-------|--------|-------------|
| `sqrt_price_x96` | `slot0()` | Encoded current price (raw uint160) |
| `tick` | `slot0()` | Current tick index |
| `liquidity` | `liquidity()` | Active in-range liquidity (raw uint128) |
| `fee` | `fee()` | Fee tier in pips (confirms on-chain value) |

Uses public RPC endpoints by default — no API key needed.  
Add `RPC_ETHEREUM`, `RPC_ARBITRUM`, `RPC_OPTIMISM`, `RPC_BASE`, `RPC_POLYGON` to `.env` to use private endpoints.

## 0. Install Dependencies

In [1]:
import subprocess, sys

def ensure(*packages):
    for pkg in packages:
        try:
            __import__(pkg.replace("-", "_"))
        except ImportError:
            print(f"[setup] Installing {pkg}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

ensure("pandas", "python-dotenv", "web3", "pyarrow")
print("All dependencies ready.")

[setup] Installing python-dotenv...
All dependencies ready.


## 1. Config

In [2]:
import os, json, time, pathlib
from datetime import date
import pandas as pd
from dotenv import load_dotenv
from web3 import Web3

load_dotenv()

# Public RPC endpoints — free, no key required
# Override any of these in .env for better reliability (Alchemy/Infura)
RPC_URLS = {
    "Ethereum": os.getenv("RPC_ETHEREUM", "https://eth.llamarpc.com"),
    "Arbitrum": os.getenv("RPC_ARBITRUM", "https://arb1.arbitrum.io/rpc"),
    "Optimism": os.getenv("RPC_OPTIMISM", "https://mainnet.optimism.io"),
    "Base":     os.getenv("RPC_BASE",     "https://mainnet.base.org"),
    "Polygon":  os.getenv("RPC_POLYGON",  "https://polygon-rpc.com"),
}

POOLS_PARQUET   = pathlib.Path("data/top_pools.parquet")
ABI_DIR         = pathlib.Path("pool_abis")
ONCHAIN_PARQUET = pathlib.Path("data/pool_onchain.parquet")
DELAY           = 0.25  # seconds between RPC calls

print("RPC endpoints:")
for chain, url in RPC_URLS.items():
    src = "(custom)" if os.getenv(f"RPC_{chain.upper()}") else "(public)"
    print(f"  {chain:<10} {src}  {url}")

RPC endpoints:
  Ethereum   (public)  https://eth.llamarpc.com
  Arbitrum   (public)  https://arb1.arbitrum.io/rpc
  Optimism   (public)  https://mainnet.optimism.io
  Base       (public)  https://mainnet.base.org
  Polygon    (public)  https://polygon-rpc.com


## 2. Load Pool List & Check Connectivity

In [ ]:
pools = pd.read_parquet(POOLS_PARQUET)

# Keep only pools that have a saved ABI
pools = pools[pools["address"].apply(lambda a: (ABI_DIR / f"{a}.json").exists())].copy()
print(f"Pools with ABI: {len(pools)} / {len(pd.read_parquet(POOLS_PARQUET))}")
print()

# Test RPC connectivity per chain
web3_clients = {}
for chain in pools["chain"].unique():
    if chain not in RPC_URLS:
        print(f"  [{chain}] no RPC URL configured — skipping")
        continue
    try:
        w3 = Web3(Web3.HTTPProvider(RPC_URLS[chain]))
        block = w3.eth.block_number   # raises if unreachable
        web3_clients[chain] = w3
        print(f"  [{chain}] connected  (latest block: {block:,})")
    except Exception as e:
        print(f"  [{chain}] FAILED to connect ({e}) — pools on this chain will be skipped")

# Filter to chains with working RPC
pools = pools[pools["chain"].isin(web3_clients)].copy()
print(f"\nReady to query {len(pools)} pools across {len(web3_clients)} chains.")

## 3. Query On-Chain State

Calls three view functions on each pool contract:
- `slot0()` → current price (sqrtPriceX96) and tick
- `liquidity()` → active in-range liquidity
- `fee()` → fee tier in pips

In [ ]:
records = []
fetched_at = str(date.today())

for chain, group in pools.groupby("chain"):
    w3 = web3_clients[chain]
    print(f"\n[{chain}]")

    for _, pool in group.iterrows():
        addr  = pool["address"]
        label = f"{pool['token0']}/{pool['token1']} {pool['fee_tier']/1e4:.2f}%"
        print(f"  {addr[:10]}...  {label}", end="", flush=True)

        try:
            abi      = json.loads((ABI_DIR / f"{addr}.json").read_text())
            contract = w3.eth.contract(address=Web3.to_checksum_address(addr), abi=abi)

            slot0     = contract.functions.slot0().call()
            liquidity = contract.functions.liquidity().call()
            fee       = contract.functions.fee().call()

            sqrt_price_x96 = slot0[0]
            tick           = slot0[1]

            # Derive human-readable price: (sqrtPriceX96 / 2^96)^2
            # Note: this is token1/token0 without decimal adjustment
            raw_price = (sqrt_price_x96 / (2 ** 96)) ** 2

            records.append({
                "address":        addr,
                "chain":          chain,
                "token0":         pool["token0"],
                "token1":         pool["token1"],
                "sqrt_price_x96": str(sqrt_price_x96),   # uint160 — store as string
                "tick":           tick,
                "liquidity":      str(liquidity),         # uint128 — store as string
                "fee":            fee,
                "price_raw":      raw_price,
                "fetched_at":     fetched_at,
            })
            print(f"  ✓  tick={tick:,}  liquidity={liquidity:.2e}  price_raw={raw_price:.6f}")

        except Exception as e:
            print(f"  ✗  {e}")

        time.sleep(DELAY)

print(f"\nQueried {len(records)} pools successfully.")


[Arbitrum]
  0x2f5e87C9...  WBTC/WETH 0.05%  ✓  tick=265,329  liquidity=4.51e+17  price_raw=333067776069.111938
  0x0E483131...  WBTC/USDC 0.05%  ✓  tick=65,361  liquidity=1.91e+12  price_raw=689.417252
  0xd13040d4...  RAIN/WETH 0.01%  ✓  tick=-124,161  liquidity=4.70e+22  price_raw=0.000004
  0xbE3aD6a5...  USDC/USDT 0.01%  ✓  tick=3  liquidity=1.25e+16  price_raw=1.000306
  0x689C96ce...  WBTC/ARB 0.30%  ✓  tick=365,296  liquidity=1.03e+18  price_raw=7308893401053627.000000
  0x80A9ae39...  WETH/GMX 1.00%  ✓  tick=57,898  liquidity=2.86e+20  price_raw=326.880120
  0x1DFc1054...  HEGIC/WETH 0.05%  ✓  tick=-118,967  liquidity=2.69e+23  price_raw=0.000007
  0x5886e46E...  USDC/THBILL 0.01%  ✓  tick=-174  liquidity=4.71e+13  price_raw=0.982817
  0x9B42809a...  WBTC/CBBTC 0.01%  ✓  tick=-21  liquidity=4.75e+12  price_raw=0.997993
  0xD8D4314f...  LON/USDC 0.30%  ✓  tick=-289,373  liquidity=1.31e+18  price_raw=0.000000
  0xdf04FA6e...  CAPX/USDC 0.01%  ✓  tick=-294,227  liquidity=5.00e+1

## 4. Save to Parquet

In [ ]:
if not records:
    print("No records collected — check RPC connectivity and ABI availability.")
else:
    df = pd.DataFrame(records)
    df.to_parquet(ONCHAIN_PARQUET, index=False, engine="pyarrow")
    print(f"Saved {len(df)} rows to {ONCHAIN_PARQUET}")
    print()
    print(df[["chain", "token0", "token1", "fee", "tick", "price_raw", "fetched_at"]])

## 5. Sanity Checks

In [ ]:
if not records:
    print("No data to check — Cell 4 produced no records.")
else:
    # Verify fee tier matches DeFiLlama data
    pools_ref = pd.read_parquet(POOLS_PARQUET)
    merged = df.merge(pools_ref[["address", "chain", "fee_tier"]], on=["address", "chain"])
    mismatches = merged[merged["fee"] != merged["fee_tier"]]

    if mismatches.empty:
        print("✓ All on-chain fee tiers match DeFiLlama data")
    else:
        print(f"✗ {len(mismatches)} fee tier mismatches:")
        print(mismatches[["chain", "token0", "token1", "fee", "fee_tier"]])

    print()
    print("Pools per chain:")
    print(df.groupby("chain")[["address"]].count().rename(columns={"address": "pools"}).to_string())